# 6-Tier Emissions Sankey Chart

**Data sources:**
- `Sector Emissions Factors` tab — B2 (total), C2 (Scope 1), D2 (Scope 2), E2 (Scope 3)
- `Tier_1 Pathway Analysis` — top 9 sectors + Other as Layer 3, with S1/S2/S3 as Layer 4
- `Tier_2 Pathway Analysis` — top 3 per Tier 1 sector + Other as Layer 5 (connected to Tier 1 Scope 3), with S1/S2/S3 as Layer 6

**Set `FILE_PATH` and `SECTOR_NAME` before running.**

In [2]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
FILE_PATH   = "TASA-EFX_rLCA_Abrasive-Material_KOR_2023_v1.0_DEMO.xlsx"  # <-- update path
SECTOR_NAME = "Abrasive Material"                                 # <-- update sector name
YEAR        = "2023"                                              # <-- update year prefix if needed

TOP_N_T1    = 9    # Top N Tier-1 sectors to show (rest → Other)
TOP_N_T2    = 3    # Top N Tier-2 sectors per Tier-1 (rest → Other)
# ─────────────────────────────────────────────────────────────────────────────

In [4]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── 1. Load top-level totals from Sector Emissions Factors ───────────────────
ef = pd.read_excel(FILE_PATH, sheet_name="Sector Emissions Factors",
                   header=0, dtype=str)
ef.columns = [str(c).strip() for c in ef.columns]

# Filter to the chosen sector row
sector_row = ef[ef.iloc[:, 0].str.strip() == SECTOR_NAME].iloc[0]

# B2 = col index 1 (0-based), C2=2, D2=3, E2=4
total_ei = float(sector_row.iloc[1])
s1_total = float(sector_row.iloc[2])
s2_total = float(sector_row.iloc[3])
s3_total = float(sector_row.iloc[4])

print(f"Sector : {SECTOR_NAME}")
print(f"Total  : {total_ei:.4f}")
print(f"Scope 1: {s1_total:.4f}")
print(f"Scope 2: {s2_total:.4f}")
print(f"Scope 3: {s3_total:.4f}")

Sector : Abrasive Material
Total  : 460.2669
Scope 1: 41.6536
Scope 2: 55.6447
Scope 3: 362.9686


In [6]:
# ── 2. Load Tier 1 Pathway Analysis ─────────────────────────────────────────
t1_raw = pd.read_excel(FILE_PATH, sheet_name="Tier_1 Pathway Analysis",
                       header=0, dtype=str)
t1_raw.columns = [str(c).strip() for c in t1_raw.columns]

# Filter to sector
sector_col = t1_raw.columns[0]        # "Sector"
t1_sec     = t1_raw[t1_raw[sector_col].str.strip() == SECTOR_NAME].copy()

# Identify column names (flexible – finds by keyword)
def find_col(df, *keywords):
    """Return first column whose name contains all keywords (case-insensitive)."""
    kws = [k.lower() for k in keywords]
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in kws):
            return c
    raise KeyError(f"No column matching {keywords} in {list(df.columns)}")

T1_SECTOR_COL = find_col(t1_sec, "tier 1", "input")
T1_TOTAL_COL  = find_col(t1_sec, "tier 1", "total")
T1_S1_COL     = find_col(t1_sec, "tier 1", "scope 1")
T1_S2_COL     = find_col(t1_sec, "tier 1", "scope 2")
T1_S3_COL     = find_col(t1_sec, "tier 1", "scope 3")

for c in [T1_TOTAL_COL, T1_S1_COL, T1_S2_COL, T1_S3_COL]:
    t1_sec[c] = pd.to_numeric(t1_sec[c], errors="coerce")

# De-dup on Tier 1 Input Sector, keep max total if duplicates exist
t1_sec = (t1_sec
          .dropna(subset=[T1_TOTAL_COL])
          .sort_values(T1_TOTAL_COL, ascending=False)
          .drop_duplicates(subset=[T1_SECTOR_COL]))

# Split Other vs non-Other, take top N non-Other, then bucket the rest
is_other_mask = t1_sec[T1_SECTOR_COL].str.strip().str.lower() == "other"
t1_non_other  = t1_sec[~is_other_mask].head(TOP_N_T1)
t1_rest       = t1_sec[~is_other_mask].iloc[TOP_N_T1:]   # sectors beyond top-N
t1_explicit_other = t1_sec[is_other_mask]

# Combine rest + explicit Other rows into a single "Other" bucket
all_other = pd.concat([t1_rest, t1_explicit_other])
if not all_other.empty:
    other_row = pd.DataFrame([{
        T1_SECTOR_COL: "Other",
        T1_TOTAL_COL:  all_other[T1_TOTAL_COL].sum(),
        T1_S1_COL:     all_other[T1_S1_COL].sum(),
        T1_S2_COL:     all_other[T1_S2_COL].sum(),
        T1_S3_COL:     all_other[T1_S3_COL].sum(),
    }])
    t1_final = pd.concat([t1_non_other, other_row]).reset_index(drop=True)
else:
    t1_final = t1_non_other.reset_index(drop=True)

print(f"Tier 1 sectors shown: {len(t1_final)}")
print(f"Tier 1 sectors sum: {t1_final[T1_TOTAL_COL].sum()}")
t1_final[[T1_SECTOR_COL, T1_TOTAL_COL]].head(12)

Tier 1 sectors shown: 10
Tier 1 sectors sum: 362.96864252354857


,Tier 1 Input Sector,Tier 1 Total Emissions Intensity
0,Basic Inorganic Compounds,38.401632
1,Other Fiber Fabrics,36.241310
2,Adhesive and Gelatin,24.677415
3,Synthetic Resin,24.205417
4,Manufacturing Equipment Repair,23.472489
5,Other Nonmetallic Mineral Products,22.073310
6,Natural and Chemical Fiber Fabrics,21.431768
7,Abrasive Material,19.253848
8,Road Haulage Transportation,17.347114
9,Other,135.864340


In [7]:
# ── 3. Load Tier 2 Pathway Analysis ─────────────────────────────────────────
t2_raw = pd.read_excel(FILE_PATH, sheet_name="Tier_2 Pathway Analysis",
                       header=0, dtype=str)
t2_raw.columns = [str(c).strip() for c in t2_raw.columns]

t2_sec = t2_raw[t2_raw.iloc[:, 0].str.strip() == SECTOR_NAME].copy()

T2_T1_COL     = find_col(t2_sec, "tier 1", "input")
T2_SECTOR_COL = find_col(t2_sec, "tier 2", "input")
T2_TOTAL_COL  = find_col(t2_sec, "tier 2", "total")
T2_S1_COL     = find_col(t2_sec, "tier 2", "scope 1")
T2_S2_COL     = find_col(t2_sec, "tier 2", "scope 2")
T2_S3_COL     = find_col(t2_sec, "tier 2", "scope 3")

for c in [T2_TOTAL_COL, T2_S1_COL, T2_S2_COL, T2_S3_COL]:
    t2_sec[c] = pd.to_numeric(t2_sec[c], errors="coerce")

t2_sec = t2_sec.dropna(subset=[T2_TOTAL_COL])

# For each Tier-1 sector in t1_final, find top-N Tier-2 sectors + Other
t2_by_t1 = {}   # dict: t1_sector_name → list of {label, total, s1, s2, s3}

for _, t1_row in t1_final.iterrows():
    t1_name = str(t1_row[T1_SECTOR_COL]).strip()

    if t1_name == "Other":
        # For the bucketed "Other" at Tier 1, we won't have direct Tier-2 rows;
        # create a single placeholder bucket equal to the Other Tier-1 scope 3
        t2_by_t1[t1_name] = [{
            "label": "Other",
            "total": float(t1_row[T1_S3_COL]),
            "s1": 0.0, "s2": 0.0, "s3": float(t1_row[T1_S3_COL])
        }]
        continue

    subset = t2_sec[t2_sec[T2_T1_COL].str.strip() == t1_name].copy()
    subset = (subset
              .sort_values(T2_TOTAL_COL, ascending=False)
              .drop_duplicates(subset=[T2_SECTOR_COL]))

    is_other_t2  = subset[T2_SECTOR_COL].str.strip().str.lower() == "other"
    top_t2       = subset[~is_other_t2].head(TOP_N_T2)
    rest_t2      = subset[~is_other_t2].iloc[TOP_N_T2:]
    explicit_t2  = subset[is_other_t2]
    all_other_t2 = pd.concat([rest_t2, explicit_t2])

    rows = []
    for _, r in top_t2.iterrows():
        rows.append({
            "label": str(r[T2_SECTOR_COL]).strip(),
            "total": float(r[T2_TOTAL_COL]),
            "s1": float(r[T2_S1_COL]),
            "s2": float(r[T2_S2_COL]),
            "s3": float(r[T2_S3_COL]),
        })
    if not all_other_t2.empty:
        rows.append({
            "label": "Other",
            "total": float(all_other_t2[T2_TOTAL_COL].sum()),
            "s1":    float(all_other_t2[T2_S1_COL].sum()),
            "s2":    float(all_other_t2[T2_S2_COL].sum()),
            "s3":    float(all_other_t2[T2_S3_COL].sum()),
        })
    t2_by_t1[t1_name] = rows

print("Tier-2 sectors per Tier-1 key:")
for k, v in t2_by_t1.items():
    print(f"  {k}: {len(v)} sectors")

Tier-2 sectors per Tier-1 key:
  Basic Inorganic Compounds: 4 sectors
  Other Fiber Fabrics: 4 sectors
  Adhesive and Gelatin: 4 sectors
  Synthetic Resin: 4 sectors
  Manufacturing Equipment Repair: 4 sectors
  Other Nonmetallic Mineral Products: 4 sectors
  Natural and Chemical Fiber Fabrics: 4 sectors
  Abrasive Material: 4 sectors
  Road Haulage Transportation: 4 sectors
  Other: 1 sectors


In [8]:
# ── 4. Assemble JSON payload ─────────────────────────────────────────────────

def pct(val, base):
    return round(val / base * 100, 2) if base else 0.0

# Layer 1 – anchor
anchor = {"label": SECTOR_NAME, "val": round(total_ei, 4)}

# Layer 2 – Scope 1/2/3
scopes = [
    {"label": "Scope 1", "val": round(s1_total, 4), "pct": pct(s1_total, total_ei), "color": "#3A86C8"},
    {"label": "Scope 2", "val": round(s2_total, 4), "pct": pct(s2_total, total_ei), "color": "#2BAE96"},
    {"label": "Scope 3", "val": round(s3_total, 4), "pct": pct(s3_total, total_ei), "color": "#1B4F8A"},
]

# Layer 3 + 4 – Tier 1 sectors and their S1/S2/S3
tier1_data = []
for _, row in t1_final.iterrows():
    t_total = float(row[T1_TOTAL_COL])
    t_s1    = float(row[T1_S1_COL])
    t_s2    = float(row[T1_S2_COL])
    t_s3    = float(row[T1_S3_COL])
    tier1_data.append({
        "label":  str(row[T1_SECTOR_COL]).strip(),
        "val":    round(t_total, 4),
        "pct":    pct(t_total, total_ei),
        "s1_val": round(t_s1, 4), "s1": pct(t_s1, total_ei),
        "s2_val": round(t_s2, 4), "s2": pct(t_s2, total_ei),
        "s3_val": round(t_s3, 4), "s3": pct(t_s3, total_ei),
    })

# Layer 5 + 6 – Tier 2 sectors (keyed by Tier-1 label)
tier2_data = {}  # key = tier1 label
for t1_label, rows in t2_by_t1.items():
    t2_list = []
    for r in rows:
        t2_list.append({
            "label":  r["label"],
            "val":    round(r["total"], 4),
            "pct":    pct(r["total"], total_ei),
            "s1_val": round(r["s1"], 4), "s1": pct(r["s1"], total_ei),
            "s2_val": round(r["s2"], 4), "s2": pct(r["s2"], total_ei),
            "s3_val": round(r["s3"], 4), "s3": pct(r["s3"], total_ei),
        })
    tier2_data[t1_label] = t2_list

payload = json.dumps({
    "sector":    SECTOR_NAME,
    "total_ei":  round(total_ei, 4),
    "scopes":    scopes,
    "tier1":     tier1_data,
    "tier2":     tier2_data,
})

print("Payload assembled. Tier1 count:", len(tier1_data),
      "  Tier2 groups:", len(tier2_data))

Payload assembled. Tier1 count: 10   Tier2 groups: 10


In [10]:
# ── 5. Render the 6-tier Sankey ──────────────────────────────────────────────

uid = "snk6t"

html = f"""
<div id="wrap-{uid}" style="font-family:'Segoe UI',Arial,sans-serif;">
  <div style="margin-bottom:10px;display:flex;gap:8px;align-items:center;">
    <span style="font-size:11px;color:#4A6A8A;font-weight:600;">Tier 1 Scope view:</span>
    <button id="btn-detail-{uid}" onclick="setMode_{uid}('detail')"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-weight:600;background:#1B4F8A;color:#fff;box-shadow:0 1px 4px rgba(0,0,0,0.15);">Broken Out</button>
    <button id="btn-summed-{uid}" onclick="setMode_{uid}('summed')"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-weight:600;background:#fff;color:#1B4F8A;">Summed</button>
    <span style="margin-left:16px;font-size:11px;color:#4A6A8A;font-weight:600;">Tier 2 sectors:</span>
    <button id="btn-t2combine-{uid}" onclick="setT2Combine_{uid}(true)"
      style="padding:5px 14px;border-radius:20px;border:none;cursor:pointer;font-size:11px;
             font-weight:600;background:#1B4F8A;color:#fff;box-shadow:0 1px 4px rgba(0,0,0,0.15);">Combined</button>
    <button id="btn-t2split-{uid}" onclick="setT2Combine_{uid}(false)"
      style="padding:5px 14px;border-radius:20px;border:2px solid #1B4F8A;cursor:pointer;font-size:11px;
             font-weight:600;background:#fff;color:#1B4F8A;">Split</button>
  </div>
  <div style="position:relative;display:inline-block;
              background:linear-gradient(135deg,#e8f0f7 0%,#f0f5f9 100%);
              padding:20px 28px 28px;border-radius:12px;
              box-shadow:0 2px 14px rgba(0,0,0,0.09);overflow-x:auto;">
    <canvas id="canvas-{uid}"></canvas>
    <div id="tip-{uid}" style="display:none;position:fixed;pointer-events:none;
      background:rgba(10,30,55,0.92);color:#fff;padding:8px 12px;border-radius:6px;
      font-size:12px;box-shadow:0 3px 12px rgba(0,0,0,0.3);line-height:1.6;
      z-index:9999;max-width:260px;"></div>
  </div>
</div>

<script>
setTimeout(function(){{

const DATA = {payload};
let currentMode = 'detail';
let combineT2 = true;

// ── Layout constants ────────────────────────────────────────────────────────
const TOP      = 72;
const LPAD     = 16;
const FULL_H   = 1100;
const BOX_GAP  = 3;
const MIN_BOX  = 10;
const LEAF_GAP = 2;

const ANC_W = 110, SC_W = 100, T1_W = 200, T1L_W = 90;
const T2_W  = 180, T2L_W = 90;
const GAP   = 120;

const ANC_X = LPAD;
const SC_X  = ANC_X + ANC_W + GAP;
const T1_X  = SC_X  + SC_W  + GAP;
const T1L_X = T1_X  + T1_W  + GAP;   // Tier1 Scope leaves
const T2_X  = T1L_X + T1L_W + GAP;   // Tier2 sectors
const T2L_X = T2_X  + T2_W  + GAP;   // Tier2 Scope leaves

function totalW() {{
  return T2L_X + T2L_W + LPAD + 10;
}}
const TOTAL_H = TOP + FULL_H + 40;

const S_COLORS = ['#3A86C8','#2BAE96','#1B4F8A'];
const T1_COLORS = [
  '#27AE8F','#2E86C1','#1D8A6E','#2471A3','#1A7A60',
  '#1F6FA3','#17725A','#1A5F8C','#136150','#174F78','#8B7355'
];
const T2_COLORS = [
  '#5B9BD5','#70C1A8','#A0C4E8','#8FD8C4','#C2DFF2',
  '#B0E8D8','#D8EEF8','#D0F0E8','#7BB8E0','#95D4BE',
  '#E8C99A','#D4B07A','#C09060','#A87040','#9A6535',
  '#8B5A2A'
];

// ── Canvas setup ────────────────────────────────────────────────────────────
const C   = document.getElementById('canvas-{uid}');
const DPR = window.devicePixelRatio || 1;
let ctx;

function resizeCanvas() {{
  const W = totalW();
  C.width  = W * DPR;
  C.height = TOTAL_H * DPR;
  C.style.width  = W + 'px';
  C.style.height = TOTAL_H + 'px';
  ctx = C.getContext('2d');
  ctx.scale(DPR, DPR);
}}

// ── Helpers ──────────────────────────────────────────────────────────────────
function hexToRgb(hex) {{
  return [parseInt(hex.slice(1,3),16),parseInt(hex.slice(3,5),16),parseInt(hex.slice(5,7),16)];
}}
function rgbaStr(hex,a) {{
  const [r,g,b]=hexToRgb(hex); return `rgba(${{r}},${{g}},${{b}},${{a}})`;
}}
function rRect(x,y,w,h,r) {{
  ctx.beginPath();
  if(ctx.roundRect) ctx.roundRect(x,y,w,h,r);
  else {{ ctx.moveTo(x+r,y); ctx.lineTo(x+w-r,y); ctx.arcTo(x+w,y,x+w,y+r,r);
    ctx.lineTo(x+w,y+h-r); ctx.arcTo(x+w,y+h,x+w-r,y+h,r);
    ctx.lineTo(x+r,y+h);   ctx.arcTo(x,y+h,x,y+h-r,r);
    ctx.lineTo(x,y+r);     ctx.arcTo(x,y,x+r,y,r); ctx.closePath(); }}
}}

function proportionalHeights(pcts, totalH, gap) {{
  const n = pcts.length, tot = pcts.reduce((a,b)=>a+b,0)||1;
  const available = totalH - gap*(n-1);
  let heights = pcts.map(p=>Math.max(MIN_BOX,(p/tot)*available));
  let diff = heights.reduce((a,b)=>a+b,0) - available;
  if (Math.abs(diff) > 0.5) {{
    const adj = heights.map((h,i)=>{{return{{i,h}}}}).filter(o=>o.h>MIN_BOX).sort((a,b)=>b.h-a.h);
    for (const o of adj) {{
      const cut = Math.min(Math.abs(diff), heights[o.i]-MIN_BOX);
      heights[o.i] -= Math.sign(diff)*cut; diff -= Math.sign(diff)*cut;
      if (Math.abs(diff)<0.5) break;
    }}
  }}
  return heights;
}}

function drawBox(x,y,w,h,color,label,pct,val,fs) {{
  fs = fs||9;
  const [r,g,b] = hexToRgb(color);
  const grad = ctx.createLinearGradient(x,y,x,y+h);
  grad.addColorStop(0,`rgba(${{Math.min(r+30,255)}},${{Math.min(g+30,255)}},${{Math.min(b+30,255)}},1)`);
  grad.addColorStop(1,color);
  ctx.fillStyle=grad; rRect(x,y,w,h,4); ctx.fill();
  ctx.strokeStyle=`rgba(${{Math.max(r-20,0)}},${{Math.max(g-20,0)}},${{Math.max(b-20,0)}},0.3)`;
  ctx.lineWidth=0.7; rRect(x,y,w,h,4); ctx.stroke();
  if(h<8) return;
  ctx.textBaseline='middle'; ctx.textAlign='left';
  const px=x+6, maxW=w-10;
  if(h>=22) {{
    ctx.font=`600 ${{fs}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.95)';
    let txt=label;
    while(ctx.measureText(txt).width>maxW&&txt.length>3) txt=txt.slice(0,-1);
    if(txt!==label) txt=txt.slice(0,-1)+'…';
    ctx.fillText(txt,px,y+h*0.35);
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.70)';
    ctx.fillText(pct+'%',px,y+h*0.68);
  }} else if(h>=14) {{
    ctx.font=`${{fs-1}}px 'Segoe UI',Arial,sans-serif`;
    ctx.fillStyle='rgba(255,255,255,0.88)';
    ctx.fillText(pct+'%',px,y+h/2);
  }}
}}

function drawRibbon(x1,y1t,y1b,x2,y2t,y2b,color,alpha) {{
  const mx = x1+(x2-x1)*0.52;
  ctx.beginPath();
  ctx.moveTo(x1,y1t); ctx.bezierCurveTo(mx,y1t,mx,y2t,x2,y2t);
  ctx.lineTo(x2,y2b); ctx.bezierCurveTo(mx,y2b,mx,y1b,x1,y1b);
  ctx.closePath(); ctx.fillStyle=rgbaStr(color,alpha); ctx.fill();
}}

function drawHeader(x,w,l1,l2) {{
  ctx.textAlign='center'; ctx.textBaseline='middle';
  ctx.fillStyle='#0D2F4F';
  ctx.font="700 10px 'Segoe UI',Arial,sans-serif";
  ctx.fillText(l1,x+w/2,TOP-(l2?24:14));
  if(l2) {{ ctx.font="400 8px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#4A6A8A'; ctx.fillText(l2,x+w/2,TOP-11); }}
  ctx.strokeStyle='rgba(26,63,100,0.15)'; ctx.lineWidth=1;
  ctx.beginPath(); ctx.moveTo(x+4,TOP-3); ctx.lineTo(x+w-4,TOP-3); ctx.stroke();
}}

// ── Tooltip ──────────────────────────────────────────────────────────────────
const tip = document.getElementById('tip-{uid}');
let hitBoxes = [];
function regHit(x,y,w,h,label,pct,val) {{
  hitBoxes.push({{x,y,w,h,tip:`<b>${{label}}</b><br>${{pct}}% of total<br>${{val.toLocaleString()}} Mg CO₂e / M USD`}});
}}
C.addEventListener('mousemove',e=>{{
  const cr = C.getBoundingClientRect();
  const mx = (e.clientX-cr.left)*(totalW()/cr.width);
  const my = (e.clientY-cr.top)*(TOTAL_H/cr.height);
  const hit = hitBoxes.find(b=>mx>=b.x&&mx<=b.x+b.w&&my>=b.y&&my<=b.y+b.h);
  if(hit) {{
    tip.innerHTML=hit.tip; tip.style.display='block';
    tip.style.left=(e.clientX+14)+'px'; tip.style.top=(e.clientY-10)+'px';
  }} else tip.style.display='none';
}});
C.addEventListener('mouseleave',()=>tip.style.display='none');

// ── Button state ─────────────────────────────────────────────────────────────
function styleBtnPair(activeId, inactiveId) {{
  [activeId, inactiveId].forEach((id,i)=>{{
    const el = document.getElementById(id);
    el.style.background = i===0 ? '#1B4F8A' : '#fff';
    el.style.color      = i===0 ? '#fff'    : '#1B4F8A';
    el.style.border     = i===0 ? 'none'    : '2px solid #1B4F8A';
  }});
}}
window.setMode_{uid} = function(mode) {{
  currentMode = mode;
  styleBtnPair(
    mode==='detail' ? 'btn-detail-{uid}' : 'btn-summed-{uid}',
    mode==='detail' ? 'btn-summed-{uid}' : 'btn-detail-{uid}'
  );
  render();
}};
window.setT2Combine_{uid} = function(combine) {{
  combineT2 = combine;
  styleBtnPair(
    combine ? 'btn-t2combine-{uid}' : 'btn-t2split-{uid}',
    combine ? 'btn-t2split-{uid}'   : 'btn-t2combine-{uid}'
  );
  render();
}};

// ── Tier-2 combine helper ────────────────────────────────────────────────────
// When combineT2=true, merge any Tier-2 sectors with the same label across
// all Tier-1 parents into a single box, summing their values.
function applyT2Combine(groups) {{
  if (!combineT2) return groups;

  // Sum all items by label across every group
  const merged = {{}};
  groups.forEach(g => {{
    g.t2items.forEach(r => {{
      if (!merged[r.label]) {{
        merged[r.label] = {{
          label: r.label, val:0, pct:0,
          s1_val:0, s1:0, s2_val:0, s2:0, s3_val:0, s3:0
        }};
      }}
      const m = merged[r.label];
      m.val+=r.val;       m.pct+=r.pct;
      m.s1_val+=r.s1_val; m.s1+=r.s1;
      m.s2_val+=r.s2_val; m.s2+=r.s2;
      m.s3_val+=r.s3_val; m.s3+=r.s3;
    }});
  }});

  const allItems = Object.values(merged).sort((a,b)=>b.val-a.val);

  // Source leaf spans from the top of the first group's leaf to the bottom of the last
  const firstLeaf = groups[0].sourceLeaf;
  const lastLeaf  = groups[groups.length-1].sourceLeaf;
  const combinedLeaf = {{
    y: firstLeaf.y,
    h: (lastLeaf.y + lastLeaf.h) - firstLeaf.y,
    color: firstLeaf.color
  }};

  return [{{t1idx: -1, t2items: allItems, sourceLeaf: combinedLeaf, t1name: 'Combined'}}];
}}

// ── Main render ──────────────────────────────────────────────────────────────
function render() {{
  hitBoxes = [];
  ctx.clearRect(0, 0, totalW(), TOTAL_H);

  // Headers
  drawHeader(ANC_X, ANC_W, 'BASELINE', 'EMISSIONS');
  drawHeader(SC_X,  SC_W,  'SCOPE',    null);
  drawHeader(T1_X,  T1_W,  'TIER 1',   'Input Sector');
  drawHeader(T1L_X, T1L_W, 'TIER 1',   currentMode==='detail' ? 'SCOPE (each)' : 'SCOPE (sum)');
  drawHeader(T2_X,  T2_W,  'TIER 2',   combineT2 ? 'Input Sector (combined)' : 'Input Sector');
  drawHeader(T2L_X, T2L_W, 'TIER 2',   currentMode==='detail' ? 'SCOPE (each)' : 'SCOPE (sum)');

  // ── ANCHOR chevron ─────────────────────────────────────────────────────────
  const chev = 16;
  const aGrad = ctx.createLinearGradient(ANC_X,TOP,ANC_X,TOP+FULL_H);
  aGrad.addColorStop(0,'#1A4A72'); aGrad.addColorStop(1,'#0D2F4F');
  ctx.fillStyle=aGrad;
  ctx.beginPath();
  ctx.moveTo(ANC_X,TOP); ctx.lineTo(ANC_X+ANC_W-chev,TOP);
  ctx.lineTo(ANC_X+ANC_W,TOP+FULL_H/2); ctx.lineTo(ANC_X+ANC_W-chev,TOP+FULL_H);
  ctx.lineTo(ANC_X,TOP+FULL_H); ctx.closePath(); ctx.fill();
  const ancCX = ANC_X+(ANC_W-chev)/2;
  ctx.textAlign='center'; ctx.textBaseline='middle';
  ctx.font="700 11px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='#fff';
  let ancLabel = DATA.sector;
  const words = ancLabel.split(' ');
  let line='', lines=[];
  words.forEach(w=>{{ if((line+' '+w).trim().length*6.5>ANC_W-20&&line) {{lines.push(line);line=w;}} else line=(line+' '+w).trim(); }});
  lines.push(line);
  lines.forEach((l,i)=>ctx.fillText(l,ancCX,TOP+FULL_H*0.38+(i-lines.length/2)*13));
  ctx.font="400 8px 'Segoe UI',Arial,sans-serif"; ctx.fillStyle='rgba(255,255,255,0.65)';
  ctx.fillText(DATA.total_ei.toLocaleString()+' Mg CO₂e/M$', ancCX, TOP+FULL_H*0.55);
  regHit(ANC_X,TOP,ANC_W,FULL_H,DATA.sector,100,DATA.total_ei);

  // ── SCOPE column (Layer 2) ──────────────────────────────────────────────────
  const scPcts    = DATA.scopes.map(s=>s.pct);
  const scHeights = proportionalHeights(scPcts,FULL_H,BOX_GAP);
  let scY_arr=[], sy=TOP;
  scHeights.forEach(h=>{{scY_arr.push(sy);sy+=h+BOX_GAP;}});

  let ancCursor=TOP;
  DATA.scopes.forEach((sc,si)=>{{
    const share=sc.pct/scPcts.reduce((a,b)=>a+b,0);
    drawRibbon(ANC_X+ANC_W-chev,ancCursor,ancCursor+share*FULL_H,
               SC_X,scY_arr[si],scY_arr[si]+scHeights[si],sc.color,0.22);
    ancCursor+=share*FULL_H;
  }});
  const scopeBoxes=[];
  DATA.scopes.forEach((sc,si)=>{{
    drawBox(SC_X,scY_arr[si],SC_W,scHeights[si],sc.color,sc.label,sc.pct,sc.val,10);
    regHit(SC_X,scY_arr[si],SC_W,scHeights[si],sc.label,sc.pct,sc.val);
    scopeBoxes.push({{y:scY_arr[si],h:scHeights[si],color:sc.color,pct:sc.pct,val:sc.val}});
  }});

  // ── TIER 1 column (Layer 3) ─────────────────────────────────────────────────
  const t1Pcts    = DATA.tier1.map(t=>t.pct);
  const t1Heights = proportionalHeights(t1Pcts,FULL_H,BOX_GAP);
  const s3box = scopeBoxes[2];
  let t1Boxes=[], t1Y=TOP;

  let s3cursor=s3box.y;
  const s3tot=s3box.pct||1;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i], col=T1_COLORS[i%T1_COLORS.length];
    const sliceH=(t.pct/s3tot)*s3box.h;
    drawRibbon(SC_X+SC_W,s3cursor,s3cursor+sliceH,T1_X,t1Y,t1Y+h,col,0.20);
    s3cursor+=sliceH; t1Y+=h+BOX_GAP;
  }});
  t1Y=TOP;
  DATA.tier1.forEach((t,i)=>{{
    const h=t1Heights[i], col=T1_COLORS[i%T1_COLORS.length];
    t1Boxes.push({{y:t1Y,h,color:col,pct:t.pct,val:t.val,label:t.label}});
    drawBox(T1_X,t1Y,T1_W,h,col,t.label,t.pct,t.val,9);
    regHit(T1_X,t1Y,T1_W,h,'Tier 1: '+t.label,t.pct,t.val);
    t1Y+=h+BOX_GAP;
  }});

  // ── TIER 1 SCOPE leaves (Layer 4) ───────────────────────────────────────────
  let t1LeafBoxes = [];

  if (currentMode === 'detail') {{
    const allLeafPcts = [];
    DATA.tier1.forEach(t=>allLeafPcts.push(t.s1,t.s2,t.s3));
    const N = allLeafPcts.length;
    const leafAvail = FULL_H - LEAF_GAP*(N-1);
    const leafTot = allLeafPcts.reduce((a,b)=>a+b,0)||1;
    const MIN_L = Math.max(4, leafAvail/N*0.20);
    let allLeafH = allLeafPcts.map(p=>Math.max(MIN_L,(p/leafTot)*leafAvail));
    const lsum = allLeafH.reduce((a,b)=>a+b,0);
    allLeafH = allLeafH.map(h=>h/lsum*leafAvail);

    let flatIdx=0, leafY=TOP;
    DATA.tier1.forEach((t,i)=>{{
      const tb = t1Boxes[i];
      const lPcts=[t.s1,t.s2,t.s3], lVals=[t.s1_val,t.s2_val,t.s3_val];
      const lhs=[allLeafH[flatIdx],allLeafH[flatIdx+1],allLeafH[flatIdx+2]];
      const ptot=lPcts.reduce((a,b)=>a+b,0)||1;
      let t1cur=tb.y;
      lhs.forEach((lh,si)=>{{
        const sliceH=(lPcts[si]/ptot)*tb.h;
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,T1L_X,leafY,leafY+lh,S_COLORS[si],0.20);
        t1cur+=sliceH;
        t1LeafBoxes.push({{y:leafY,h:lh,color:S_COLORS[si],pct:lPcts[si],val:lVals[si],t1idx:i,si}});
        leafY+=lh+LEAF_GAP;
      }});
      flatIdx+=3;
    }});
    t1LeafBoxes.forEach(b=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][b.si];
      drawBox(T1L_X,b.y,T1L_W,b.h,b.color,lbl,b.pct,b.val,8);
      regHit(T1L_X,b.y,T1L_W,b.h,DATA.tier1[b.t1idx].label+' · '+lbl,b.pct,b.val);
    }});

  }} else {{
    // Summed mode
    const smPcts = [DATA.tier1.reduce((a,t)=>a+t.s1,0),
                    DATA.tier1.reduce((a,t)=>a+t.s2,0),
                    DATA.tier1.reduce((a,t)=>a+t.s3,0)];
    const smVals = [DATA.tier1.reduce((a,t)=>a+t.s1_val,0),
                    DATA.tier1.reduce((a,t)=>a+t.s2_val,0),
                    DATA.tier1.reduce((a,t)=>a+t.s3_val,0)];
    const smH    = proportionalHeights(smPcts,FULL_H,BOX_GAP);
    DATA.tier1.forEach((t,i)=>{{
      const tb=t1Boxes[i];
      const lPcts=[t.s1,t.s2,t.s3]; const ptot=lPcts.reduce((a,b)=>a+b,0)||1;
      let t1cur=tb.y; let bucketY=TOP;
      smH.forEach((bh,si)=>{{
        const sliceH=(lPcts[si]/ptot)*tb.h;
        drawRibbon(T1_X+T1_W,t1cur,t1cur+sliceH,T1L_X,bucketY,bucketY+bh,S_COLORS[si],0.12);
        t1cur+=sliceH; bucketY+=bh+BOX_GAP;
      }});
    }});
    let smY=TOP;
    smH.forEach((h,si)=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][si];
      drawBox(T1L_X,smY,T1L_W,h,S_COLORS[si],lbl,smPcts[si],smVals[si],9);
      regHit(T1L_X,smY,T1L_W,h,'Tier 1 '+lbl+' (all)',smPcts[si],smVals[si]);
      t1LeafBoxes.push({{y:smY,h,color:S_COLORS[si],pct:smPcts[si],val:smVals[si],t1idx:-1,si}});
      smY+=h+BOX_GAP;
    }});
  }}

  // ── TIER 2 sectors (Layer 5) ─────────────────────────────────────────────────
  // Build raw groups: one per Tier-1 sector, sourced from its Scope 3 leaf
  const t2RawGroups = [];

  if (currentMode === 'detail') {{
    const s3Leaves = t1LeafBoxes.filter(b=>b.si===2);
    s3Leaves.forEach((leaf,i)=>{{
      const t1name = DATA.tier1[leaf.t1idx].label;
      t2RawGroups.push({{t1idx:leaf.t1idx, t2items:DATA.tier2[t1name]||[], sourceLeaf:leaf, t1name}});
    }});
  }} else {{
    // Summed mode: carve the single Scope-3 bucket proportionally per Tier-1
    const s3Leaf = t1LeafBoxes.find(b=>b.si===2);
    const s3Total = DATA.tier1.reduce((a,x)=>a+x.s3,0)||1;
    let cursor = s3Leaf.y;
    DATA.tier1.forEach((t,i)=>{{
      const share = t.s3 / s3Total;
      const vLeaf = {{y:cursor, h:s3Leaf.h*share, color:s3Leaf.color}};
      t2RawGroups.push({{t1idx:i, t2items:DATA.tier2[t.label]||[], sourceLeaf:vLeaf, t1name:t.label}});
      cursor += s3Leaf.h * share;
    }});
  }}

  // Apply combine toggle
  const t2Groups = applyT2Combine(t2RawGroups);

  // Size all Tier-2 sector boxes proportionally across FULL_H
  const allT2Pcts = [];
  t2Groups.forEach(g=>g.t2items.forEach(r=>allT2Pcts.push(r.pct)));
  const totT2Pcts = allT2Pcts.reduce((a,b)=>a+b,0)||1;
  const t2AvailH  = FULL_H - LEAF_GAP*(allT2Pcts.length-1);
  const MIN_T2    = Math.max(4, t2AvailH/Math.max(allT2Pcts.length,1)*0.15);
  let allT2H = allT2Pcts.map(p=>Math.max(MIN_T2,(p/totT2Pcts)*t2AvailH));
  const t2Hsum = allT2H.reduce((a,b)=>a+b,0);
  allT2H = allT2H.map(h=>h/t2Hsum*t2AvailH);

  let t2Y=TOP, flatT2=0;
  const t2BoxList = [];

  t2Groups.forEach((g,gi)=>{{
    const src = g.sourceLeaf;
    let srcCursor = src.y;
    const srcTot = g.t2items.reduce((a,r)=>a+r.pct,0)||1;

    g.t2items.forEach((r,ri)=>{{
      const h   = allT2H[flatT2];
      const col = T2_COLORS[(gi*4+ri)%T2_COLORS.length];
      const sliceH = (r.pct/srcTot)*src.h;
      drawRibbon(T1L_X+T1L_W, srcCursor, srcCursor+sliceH,
                 T2_X, t2Y, t2Y+h, col, 0.20);
      srcCursor += sliceH;
      t2BoxList.push({{y:t2Y, h, color:col, t2:r, gi, ri}});
      t2Y += h+LEAF_GAP; flatT2++;
    }});
  }});

  t2BoxList.forEach(b=>{{
    drawBox(T2_X,b.y,T2_W,b.h,b.color,b.t2.label,b.t2.pct,b.t2.val,8);
    regHit(T2_X,b.y,T2_W,b.h,'Tier 2: '+b.t2.label,b.t2.pct,b.t2.val);
  }});

  // ── TIER 2 SCOPE leaves (Layer 6) ─────────────────────────────────────────
  // Mirror the Tier-1 scope toggle: detail = S1/S2/S3 per box, summed = 3 global buckets
  if (currentMode === 'detail') {{
    const allT2LPcts = [];
    t2BoxList.forEach(b=>allT2LPcts.push(b.t2.s1,b.t2.s2,b.t2.s3));
    const t2LAvail = FULL_H - LEAF_GAP*(allT2LPcts.length-1);
    const t2LTot   = allT2LPcts.reduce((a,b)=>a+b,0)||1;
    const MIN_T2L  = Math.max(3, t2LAvail/Math.max(allT2LPcts.length,1)*0.10);
    let allT2LH = allT2LPcts.map(p=>Math.max(MIN_T2L,(p/t2LTot)*t2LAvail));
    const t2LHsum = allT2LH.reduce((a,b)=>a+b,0);
    allT2LH = allT2LH.map(h=>h/t2LHsum*t2LAvail);

    let t2LY=TOP, flatL=0;
    t2BoxList.forEach(b=>{{
      const lPcts=[b.t2.s1,b.t2.s2,b.t2.s3];
      const lVals=[b.t2.s1_val,b.t2.s2_val,b.t2.s3_val];
      const ptot=lPcts.reduce((a,x)=>a+x,0)||1;
      let t2cur=b.y;
      lPcts.forEach((p,si)=>{{
        const lh=allT2LH[flatL];
        const sliceH=(p/ptot)*b.h;
        drawRibbon(T2_X+T2_W,t2cur,t2cur+sliceH,T2L_X,t2LY,t2LY+lh,S_COLORS[si],0.20);
        drawBox(T2L_X,t2LY,T2L_W,lh,S_COLORS[si],['S1','S2','S3'][si],p,lVals[si],7);
        regHit(T2L_X,t2LY,T2L_W,lh,'Tier 2: '+b.t2.label+' · '+['Scope 1','Scope 2','Scope 3'][si],p,lVals[si]);
        t2cur+=sliceH; t2LY+=lh+LEAF_GAP; flatL++;
      }});
    }});

  }} else {{
    // Summed mode: collapse all Tier-2 S1/S2/S3 into 3 global buckets
    const sm2Pcts = [t2BoxList.reduce((a,b)=>a+b.t2.s1,0),
                     t2BoxList.reduce((a,b)=>a+b.t2.s2,0),
                     t2BoxList.reduce((a,b)=>a+b.t2.s3,0)];
    const sm2Vals = [t2BoxList.reduce((a,b)=>a+b.t2.s1_val,0),
                     t2BoxList.reduce((a,b)=>a+b.t2.s2_val,0),
                     t2BoxList.reduce((a,b)=>a+b.t2.s3_val,0)];
    const sm2H    = proportionalHeights(sm2Pcts, FULL_H, BOX_GAP);

    // Ribbons: each Tier-2 box fans into the 3 summed buckets
    let bucketTops = []; let bCursor=TOP;
    sm2H.forEach(h=>{{bucketTops.push(bCursor); bCursor+=h+BOX_GAP;}});
    t2BoxList.forEach(b=>{{
      const lPcts=[b.t2.s1,b.t2.s2,b.t2.s3];
      const ptot=lPcts.reduce((a,x)=>a+x,0)||1;
      let t2cur=b.y;
      lPcts.forEach((p,si)=>{{
        const sliceH=(p/ptot)*b.h;
        drawRibbon(T2_X+T2_W,t2cur,t2cur+sliceH,T2L_X,bucketTops[si],bucketTops[si]+sm2H[si],S_COLORS[si],0.12);
        t2cur+=sliceH;
      }});
    }});

    // Draw the 3 summed boxes
    let bY=TOP;
    sm2H.forEach((h,si)=>{{
      const lbl=['Scope 1','Scope 2','Scope 3'][si];
      drawBox(T2L_X,bY,T2L_W,h,S_COLORS[si],lbl,sm2Pcts[si],sm2Vals[si],9);
      regHit(T2L_X,bY,T2L_W,h,'Tier 2 '+lbl+' (all)',sm2Pcts[si],sm2Vals[si]);
      bY+=h+BOX_GAP;
    }});
  }}

  // Footer
  ctx.font="400 8px 'Segoe UI',Arial,sans-serif";
  ctx.fillStyle='rgba(80,110,140,0.60)';
  ctx.textAlign='left'; ctx.textBaseline='top';
  ctx.fillText('Emissions Intensity (Mg CO₂e per Million USD) · '+DATA.sector, ANC_X, TOP+FULL_H+10);
}}

resizeCanvas();
render();
}}, 150);
</script>
"""

display(HTML(html))